# Section 2 — Inspecting an A2A agent

**⏱ About 7 minutes.**

**First, in a terminal:**

```bash
./scripts/run_a2a_server.sh
```

Then run the cells below, top to bottom.

### The idea in one line

MCP connected an agent to *tools*. A2A connects an agent to *another agent*.

| | A tool | A peer agent |
|---|---|---|
| What it is | a function | its own model, instructions and tools |
| What you do | **call** it | **delegate** to it |
| You get back | a return value | a task it worked on |

In [1]:
import json
import httpx

BASE = "http://localhost:8001/a2a/billing_agent"

# a2a-sdk 0.3.x served the card at /.well-known/agent.json; 1.x serves it at
# /.well-known/agent-card.json. In real code use the ADK constant
# AGENT_CARD_WELL_KNOWN_PATH rather than hardcoding either.
CARD_URL = f"{BASE}/.well-known/agent-card.json"
print(CARD_URL)

http://localhost:8001/a2a/billing_agent/.well-known/agent-card.json


## 1. The agent card

This is all of A2A discovery: **a JSON file at a well-known URL.**

No registry, no service mesh, no broker. If you can GET this file, you can work
with the agent. Skim the output — we pull out the fields that matter in step 2.

In [2]:
card = httpx.get(CARD_URL).json()
print(json.dumps(card, indent=2))

{
  "name": "billing_agent",
  "description": "Billing specialist for the IT support desk. Handles invoice questions, duplicate or unexpected charges, refunds, and plan changes.",
  "supportedInterfaces": [
    {
      "url": "http://localhost:8001/a2a/billing_agent",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "0.3.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": false,
    "pushNotifications": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "billing_enquiry",
      "name": "Billing Enquiry",
      "description": "Investigate invoice questions, duplicate charges, refunds and credits, and explain the resolution path.",
      "tags": [
        "billing",
        "invoice",
        "refund",
        "charge",
        "finance"
      ]
    }
  ],
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "url": "http://localhost:8001/a2a/billing_agent"

> **What just happened**
>
> A plain `GET` returned everything needed to work with an agent we had never
> talked to before. Discovery is a static file, so any language that can do
> HTTP can join in.
>
> **The failure you will actually hit:** a missing or misnamed `agent.json`. The
> server then publishes nothing and fails silently.

## 2. Reading the card the way a caller does

Three fields carry the weight:

- **`skills`** — what it can do, in plain language. A calling agent's model
  reads this to decide whether to delegate here.
- **`capabilities`** — protocol features, such as streaming.
- **`url`** — where to actually send work.

In [3]:
# The card carries the RPC url in two places at once: "supportedInterfaces"
# (protocol 1.0) and a top-level "url" (legacy 0.3). Read the modern one first
# and fall back -- a real client library does this for you.

def resolve_rpc_url(card: dict) -> str:
    interfaces = card.get("supportedInterfaces") or []
    if interfaces and interfaces[0].get("url"):
        return interfaces[0]["url"]
    return card["url"]  # legacy 0.3 placement


RPC_URL = resolve_rpc_url(card)

print("name:  ", card["name"])
print("rpc:   ", RPC_URL)
print("caps:  ", card.get("capabilities"))
print()
for skill in card.get("skills", []):
    print(f"- {skill['id']}: {skill['description']}")
    print(f"  tags: {', '.join(skill.get('tags', []))}")

name:   billing_agent
rpc:    http://localhost:8001/a2a/billing_agent
caps:   {'streaming': False, 'pushNotifications': False}

- billing_enquiry: Investigate invoice questions, duplicate charges, refunds and credits, and explain the resolution path.
  tags: billing, invoice, refund, charge, finance


> **What just happened**
>
> We pulled out the three fields a caller needs.
>
> **Why it matters:** `skills` is the A2A equivalent of an MCP docstring — vague
> description, no delegation.
>
> **Watch the `url`.** It is absolute, which is why we keep all traffic inside
> the Codespace: behind a forwarded URL the card still advertises `localhost`,
> so discovery succeeds and every call fails.

## 3. Sending work over the wire

**Like an MCP tool call, an A2A request is two things:**

1. a **method** — always `message/send`, whatever the agent does
2. a **message** — a role, an id, and your text

Notice what is *not* in the request: there is no `billing` method. Every A2A
agent answers the same call.

In [5]:
import uuid

QUESTION = "What is the refund policy for annual plans?"

payload = {
    "jsonrpc": "2.0",
    "id": str(uuid.uuid4()),
    "method": "message/send",
    "params": {
        "message": {
            "role": "user",
            "messageId": str(uuid.uuid4()),
            "parts": [{"kind": "text", "text": QUESTION}],
        }
    },
}

response = httpx.post(RPC_URL, json=payload, timeout=120.0)
print("status:", response.status_code)


def text_parts(node, seen=None):
    """Collect the agent's text, wherever it sits in the reply."""
    seen = {} if seen is None else seen
    if isinstance(node, dict):
        if node.get("role") == "user":
            return  # the task history echoes our own question back; skip it
        text = node.get("text")
        if node.get("kind") == "text" and text and text not in seen:
            seen[text] = True
            yield text
        for value in node.values():
            yield from text_parts(value, seen)
    elif isinstance(node, list):
        for value in node:
            yield from text_parts(value, seen)


body = response.json()

print("\nasked:", QUESTION)
print("\nthe agent replied:\n")
# The answer arrives in "artifacts"; "history" repeats it, hence the dedupe.
for part in text_parts(body.get("result", {})):
    print(part)

print("\n--- raw envelope (truncated) ---")
print(json.dumps(body, indent=2)[:1200])


status: 200

asked: What is the refund policy for annual plans?

the agent replied:

The knowledge base does not specify a general refund policy for annual plans. It only confirms that duplicate or mid-cycle plan-change charges may receive a prorated credit; approved refunds to the original payment method take **5–7 days**.

**Next step:** Escalate the annual-plan refund request to Finance for eligibility review. Expect the payment refund within **5–7 days after approval**.

--- raw envelope (truncated) ---
{
  "id": "e232a4b3-c58d-4a2b-aaf6-7cb5a67875f9",
  "jsonrpc": "2.0",
  "result": {
    "artifacts": [
      {
        "artifactId": "5b210ea0-ef26-46b8-8dbf-db4c7458a7a9",
        "parts": [
          {
            "kind": "text",
            "text": "The knowledge base does not specify a general refund policy for annual plans. It only confirms that duplicate or mid-cycle plan-change charges may receive a prorated credit; approved refunds to the original payment method take **5\u20

> **What just happened**
>
> We sent one sentence over HTTP and a *different process* — with its own model
> and instructions — answered. This is byte-for-byte what `triage_agent` sends
> when it delegates.
>
> **Why it matters:** the specialty lives in the **card**, not in method names.
> Every agent exposes the identical `message/send` verb, so adding a tenth agent
> needs no new client code.
>
> The reply came back as a *task*, not a string, because delegation can be
> long-running.

## 4. Now watch an agent do it

> **⭐ Optional — run this live only if you have time.** It needs a second
> server and a model round trip. Everything above already proves the protocol;
> this shows what it feels like. Full steps: `docs/02-lab-a2a.md`.

So far *we* have been the caller. Now let a model do it.

```bash
./scripts/run_web.sh
```

Open http://127.0.0.1:8002, pick **triage_agent**, and ask:

> *Why was I charged twice this month?*

Triage decides this is billing and hands off to `billing_agent` — a different
process it knows about only through the card you read above.

Then open `src/helpdesk/a2a/local/triage_agent/agent.py`: the remote agent is
declared in three lines and dropped into `sub_agents=[...]` beside local ones.

> **Why it matters:** from the model's point of view there is no difference
> between a local sub-agent and one on another machine. Teams can ship agents
> independently, in different languages, and still compose them.

---

## Recap

| We did | The lesson |
|---|---|
| GET the agent card | discovery is a static JSON file at a known URL |
| Read `skills` | it is the prompt another model reads |
| Read `url` | it is absolute, which breaks under URL forwarding |
| POST `message/send` | one verb for every agent |
| Watched the UI | remote agents look local to the model |

Next: `docs/03-lab-interop.md` — an **optional bonus lab** where one agent uses
MCP and A2A at the same time. Sections 1 and 2 are the core workshop.